**Bloque 1 — instalar GLiNER**

In [2]:
!pip install gliner pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.8/207.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 64.0 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.10.2
    Uninstalling transformers-5.10.2:
      Successfully uninstalled transformers-5.10.2


**Bloque 2 — imports**

In [4]:
import pandas as pd
from tqdm import tqdm
from gliner import GLiNER

tqdm.pandas()

**Bloque 3 — subir archivo de embargos**

In [5]:
from google.colab import files

uploaded = files.upload()

Saving embargos_limpios.csv to embargos_limpios.csv


**Bloque 4 — cargar dataset**

In [6]:
df_embargos = pd.read_csv("embargos_limpios.csv")

print("Shape:", df_embargos.shape)
df_embargos.head()

Shape: (627, 14)


,id,texto_ocr,nombre,clasificacion,tiene_html_detectado,patrones_html_detectados,estado_texto_ocr,largo_texto_ocr,largo_original,texto_limpio,texto_normalizado_busqueda,largo_limpio,texto_markdown,diferencia_largo
0,494145,ii\n4373-350\no.gestona.profesionalU gmail.com...,Embargo - usuario,Embargo,False,[],ok,3539,3541,ii\n4373-350\no.gestona.profesionalU gmail.com...,ii 4373-350 o.gestona.profesionalu gmail.com a...,3539,# Documento legal\n\n## Metadatos\n\n* ID: 494...,2
1,494146,R\n\n012 E\n\nercado Poder Judicial de la Naci...,Embargo - usuario,Embargo,False,[],ok,3124,3125,R\n\n012 E\n\nercado Poder Judicial de la Naci...,r 012 e ercado poder judicial de la nacion juz...,3124,# Documento legal\n\n## Metadatos\n\n* ID: 494...,1
2,494147,|\nTEXTO Y PATOS DE LA NOTIFICACIÓN LE!\n\nUsu...,Embargo - usuario,Embargo,False,[],ok,5167,5169,|\nTEXTO Y PATOS DE LA NOTIFICACIÓN LE!\n\nUsu...,| texto y patos de la notificacion le! usuario...,5166,# Documento legal\n\n## Metadatos\n\n* ID: 494...,3
3,494150,02lNE\n\nOFICIO e y |\n\nCiudad Autónoma de Bu...,Embargo - usuario,Embargo,False,[],ok,3819,3820,02lNE\n\nOFICIO e y |\n\nCiudad Autónoma de Bu...,02lne oficio e y | ciudad autonoma de buenos a...,3818,# Documento legal\n\n## Metadatos\n\n* ID: 494...,2
4,494151,SDE LA NOTIFICACIÓN (\nFRAPPA LETICIA MABEL - ...,Embargo - usuario,Embargo,False,[],ok,8876,8878,SDE LA NOTIFICACIÓN (\nFRAPPA LETICIA MABEL - ...,sde la notificacion ( frappa leticia mabel - 2...,8875,# Documento legal\n\n## Metadatos\n\n* ID: 494...,3


Validación:

In [5]:
df_embargos["clasificacion"].value_counts()

,count
clasificacion,
Embargo,627


**6. Cargar GLiNER**  
Usamos el modelo que aparece en el Colab de guia

In [6]:
model = GLiNER.from_pretrained("urchade/gliner_multi_pii-v1")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

**7. Contar tokens con el tokenizer de GLiNER**  

Primero intentamos acceder al tokenizer interno:

In [7]:
tokenizer = model.data_processor.transformer_tokenizer

Definimos funcion

In [8]:
def contar_tokens_gliner(texto):
    if pd.isna(texto):
        return 0

    texto = str(texto)

    tokens = tokenizer(
        texto,
        add_special_tokens=True,
        truncation=False
    )

    return len(tokens["input_ids"])

Ahora aplicamos sobre el texto limpio:

In [9]:
df_embargos["gliner_tokens"] = df_embargos["texto_limpio"].progress_apply(contar_tokens_gliner)

100%|██████████| 627/627 [00:02<00:00, 239.91it/s]


**8. Sacar media, mínimo y máximo**

In [10]:
df_embargos["gliner_tokens"].describe()

,gliner_tokens
count,627.000000
mean,1128.094099
std,513.130490
min,149.000000
25%,771.500000
50%,1055.000000
75%,1378.000000
max,3517.000000


También podés sacar solo lo pedido:

In [11]:
resumen_tokens_embargos = df_embargos["gliner_tokens"].agg(["count", "mean", "min", "max"])

resumen_tokens_embargos

,gliner_tokens
count,627.000000
mean,1128.094099
min,149.000000
max,3517.000000


En formato mas claro:

In [12]:
print("Cantidad de embargos:", df_embargos.shape[0])
print("Media de tokens:", round(df_embargos["gliner_tokens"].mean(), 2))
print("Mínimo de tokens:", df_embargos["gliner_tokens"].min())
print("Máximo de tokens:", df_embargos["gliner_tokens"].max())

Cantidad de embargos: 627
Media de tokens: 1128.09
Mínimo de tokens: 149
Máximo de tokens: 3517


**9. Explorar tokens por subtipo de embargo**  

Tu archivo tiene columna nombre, por ejemplo:

Agrupar por nombre.

Esto sirve para la tarea:

exploración de tipo de datos para ver cuántos tokens consumen

En tu caso sería: qué subtipo de embargo consume más tokens.

In [13]:
df_embargos.groupby("nombre")["gliner_tokens"].agg(["count", "mean", "min", "max"]).sort_values("mean", ascending=False)

,count,mean,min,max
nombre,,,,
Embargo - Empleado,4,1754.750000,836,2376
Embargo - usuario,623,1124.070626,149,3517


**10. Ver embargos más largos**

In [14]:
df_embargos.sort_values("gliner_tokens", ascending=False)[
    ["id", "nombre", "clasificacion", "gliner_tokens", "largo_limpio"]
].head(20)

,id,nombre,clasificacion,gliner_tokens,largo_limpio
35,496830,Embargo - usuario,Embargo,3517,10926
475,538037,Embargo - usuario,Embargo,3166,9437
4,494151,Embargo - usuario,Embargo,3050,8875
168,514558,Embargo - usuario,Embargo,3042,9372
150,513537,Embargo - usuario,Embargo,2971,8140
66,502593,Embargo - usuario,Embargo,2917,9085
394,531053,Embargo - usuario,Embargo,2762,8751
368,529783,Embargo - usuario,Embargo,2726,8689
235,522288,Embargo - usuario,Embargo,2655,7517
396,531070,Embargo - usuario,Embargo,2648,8398


Y revisar uno:
Esto te ayuda a detectar si hay documentos demasiado largos para GLiNER.

In [15]:
id_largo = df_embargos.sort_values("gliner_tokens", ascending=False).iloc[0]["id"]

row = df_embargos[df_embargos["id"] == id_largo].iloc[0]

print("ID:", row["id"])
print("Tokens:", row["gliner_tokens"])
print("Nombre:", row["nombre"])

print(row["texto_limpio"][:4000])

ID: 496830
Tokens: 3517
Nombre: Embargo - usuario
UPREMA DE JUSTICIA Fecha Impresión
| 26/12/2025 - 08:53:52

1125. | A ecoroasrass MN TAR

DEN 0

| JEICIO CASILLERO VIRTUAL CON FIRMA DIGITAL

ito: [2/12/2025 00:00
bsitada en el/los domicilio/s digitales:
SARMIENTO, JULIO ISAAC-ACTOR

PODER JUDICIAL DE TUCUMÁN

LL

H106038878557

¡AL CAPITAL ES mercado
> libre

Stión Asociada en Documentos y Locaciones N* 3

"lf375/25 0 8 ENE 2026

UAM RECEPCION

Jutgalib Civillen Documentos y Locaciones 1H

San 1
|
AL SR,

Ss

S/D

JUICI
7375/23.

En los
Locadi
Coordi
Fradejas, Dr

e Tucumán, 11 de diciembre de 2025.

LITADO DE

PAGO (MERCADO LIBRE SRL)

ARMIENTO JULIO ISAAC e/ EL MALIK S.A.S. s/ COBRO EJECUTIVO - Expte. n?

del rubro que se tramitan por ante esta Oficina de Gestión Asociada en Documentos y
* 3, del Centro Judicial Capital, a cargo del Director. Dr. Hector Bazán Perez,
as, Dr. Jose Manuel Torrego, Dr. Fernando Jose Lucas Filgueira, Dra. Elena Carolina
Diego Martin Gonzalez, se ha disp

**11. Labels iniciales para embargos**

Para empezar con GLiNER, yo usaría labels específicos de embargos:

In [16]:
labels_embargo = [
    "persona embargada",
    "empresa embargada",
    "DNI",
    "CUIT",
    "CUIL",
    "monto",
    "monto embargado",
    "monto costas",
    "CBU",
    "CVU",
    "cuenta judicial",
    "banco",
    "juzgado",
    "expediente",
    "caratula",
    "fecha",
    "email",
    "destinatario",
    "plazo"
]

**12. Probar GLiNER con un embargo**

In [17]:
row = df_embargos.sample(1, random_state=42).iloc[0]

text = row["texto_limpio"]

entities = model.predict_entities(text, labels_embargo)

print("ID:", row["id"])
print("Tokens:", row["gliner_tokens"])

for entity in entities:
    print(entity["text"], "=>", entity["label"], "| score:", round(entity.get("score", 0), 3))

ID: 544832
Tokens: 562
Expediente. Nro. 21903/14 => expediente | score: 0.861
Juzgado Nacional de Primera Instancia del Trabajo Nro. 35 => juzgado | score: 0.927
FAENADORA ARGENTINA S.A. => empresa embargada | score: 0.93
33-71237566-9 => CUIT | score: 0.885
$ 31.177,68.- => monto | score: 0.759
$9.000.- => monto | score: 0.741
IMÍOreSES Y COSÍAS => monto costas | score: 0.519
BANCO DE LA CIUDAD DE
BUENOS AIRES => banco | score: 0.936
30-99903208-3 => CUIT | score: 0.527


**13. Guardar entidades en formato tabla**

Después de probar uno o dos, podés probar con 10:

In [18]:
def extraer_entidades_documento(row, labels):
    text = str(row["texto_limpio"])

    try:
        entities = model.predict_entities(text, labels)

        resultados = []

        for ent in entities:
            resultados.append({
                "id": row["id"],
                "nombre": row["nombre"],
                "clasificacion": row["clasificacion"],
                "gliner_tokens": row["gliner_tokens"],
                "entidad_texto": ent.get("text"),
                "entidad_label": ent.get("label"),
                "score": ent.get("score")
            })

        return resultados

    except Exception as e:
        return [{
            "id": row["id"],
            "nombre": row["nombre"],
            "clasificacion": row["clasificacion"],
            "gliner_tokens": row["gliner_tokens"],
            "entidad_texto": None,
            "entidad_label": "ERROR",
            "score": None,
            "error": str(e)
        }]

Aplicar a muestra:

In [19]:
muestra_embargos = df_embargos.sample(10, random_state=42)

todos_resultados = []

for _, row in tqdm(muestra_embargos.iterrows(), total=len(muestra_embargos)):
    todos_resultados.extend(extraer_entidades_documento(row, labels_embargo))

df_entidades_embargos_muestra = pd.DataFrame(todos_resultados)

df_entidades_embargos_muestra.head(50)

 10%|█         | 1/10 [00:05<00:51,  5.77s/it]/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 997 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
 20%|██        | 2/10 [00:11<00:45,  5.67s/it]/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 987 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
 30%|███       | 3/10 [00:17<00:41,  5.99s/it]/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 738 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
 40%|████      | 4/10 [00:22<00:32,  5.37s/it]/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:417: UserWarni

,id,nombre,clasificacion,gliner_tokens,entidad_texto,entidad_label,score
0,544832,Embargo - usuario,Embargo,562,Expediente. Nro. 21903/14,expediente,0.861329
1,544832,Embargo - usuario,Embargo,562,Juzgado Nacional de Primera Instancia del Trab...,juzgado,0.927369
2,544832,Embargo - usuario,Embargo,562,FAENADORA ARGENTINA S.A.,empresa embargada,0.929711
3,544832,Embargo - usuario,Embargo,562,33-71237566-9,CUIT,0.884942
4,544832,Embargo - usuario,Embargo,562,"$ 31.177,68.-",monto,0.758830
5,544832,Embargo - usuario,Embargo,562,$9.000.-,monto,0.740670
6,544832,Embargo - usuario,Embargo,562,IMÍOreSES Y COSÍAS,monto costas,0.518556
7,544832,Embargo - usuario,Embargo,562,BANCO DE LA CIUDAD DE\nBUENOS AIRES,banco,0.935519
8,544832,Embargo - usuario,Embargo,562,30-99903208-3,CUIT,0.527177
9,546230,Embargo - usuario,Embargo,1542,JUZGADO CIVIL 7,juzgado,0.590764


Guardar:

In [20]:
df_entidades_embargos_muestra.to_csv("entidades_embargos_muestra_gliner.csv", index=False, encoding="utf-8-sig")

Descargar:

In [21]:
files.download("entidades_embargos_muestra_gliner.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Resultado

In [22]:
df_embargos.shape
df_embargos["gliner_tokens"].describe()
df_embargos.sort_values("gliner_tokens", ascending=False)[["id", "nombre", "gliner_tokens"]].head(10)
df_entidades_embargos_muestra.head(50)

,id,nombre,clasificacion,gliner_tokens,entidad_texto,entidad_label,score
0,544832,Embargo - usuario,Embargo,562,Expediente. Nro. 21903/14,expediente,0.861329
1,544832,Embargo - usuario,Embargo,562,Juzgado Nacional de Primera Instancia del Trab...,juzgado,0.927369
2,544832,Embargo - usuario,Embargo,562,FAENADORA ARGENTINA S.A.,empresa embargada,0.929711
3,544832,Embargo - usuario,Embargo,562,33-71237566-9,CUIT,0.884942
4,544832,Embargo - usuario,Embargo,562,"$ 31.177,68.-",monto,0.758830
5,544832,Embargo - usuario,Embargo,562,$9.000.-,monto,0.740670
6,544832,Embargo - usuario,Embargo,562,IMÍOreSES Y COSÍAS,monto costas,0.518556
7,544832,Embargo - usuario,Embargo,562,BANCO DE LA CIUDAD DE\nBUENOS AIRES,banco,0.935519
8,544832,Embargo - usuario,Embargo,562,30-99903208-3,CUIT,0.527177
9,546230,Embargo - usuario,Embargo,1542,JUZGADO CIVIL 7,juzgado,0.590764


In [23]:
print("Cantidad de documentos procesados:")
print(df_entidades_embargos_muestra["id"].nunique())

print("\nCantidad total de entidades detectadas:")
print(df_entidades_embargos_muestra.shape[0])

print("\nEntidades por etiqueta:")
print(df_entidades_embargos_muestra["entidad_label"].value_counts())

print("\nScore promedio por etiqueta:")
print(
    df_entidades_embargos_muestra
    .groupby("entidad_label")["score"]
    .agg(["count", "mean", "min", "max"])
    .sort_values("mean", ascending=False)
)

Cantidad de documentos procesados:
10

Cantidad total de entidades detectadas:
72

Entidades por etiqueta:
entidad_label
juzgado              14
expediente           10
monto                 9
fecha                 9
destinatario          7
CUIT                  7
banco                 2
monto costas          2
DNI                   2
email                 2
persona embargada     2
cuenta judicial       2
empresa embargada     1
CBU                   1
CVU                   1
caratula              1
Name: count, dtype: int64

Score promedio por etiqueta:
                   count      mean       min       max
entidad_label                                         
CBU                    1  0.983295  0.983295  0.983295
banco                  2  0.940338  0.935519  0.945156
empresa embargada      1  0.929711  0.929711  0.929711
caratula               1  0.886669  0.886669  0.886669
juzgado               14  0.811268  0.590764  0.966946
expediente            10  0.783782  0.544367  0.925378

**CONCLUSION**  
Se realizó una prueba inicial de GLiNER sobre 10 embargos. El modelo detectó 72 entidades, principalmente juzgados, expedientes, montos, fechas, destinatarios y CUIT. Las entidades más confiables fueron CBU, banco, empresa embargada, carátula, juzgado y expediente. Algunas etiquetas como persona embargada, monto costas y destinatario tuvieron scores más bajos, por lo que requieren revisión o ajuste de labels.

Probar GLiNER en **1 embargo corto, 1 promedio y 1 largo**

Esta prueba es clave. Sirve para ver si el rendimiento cambia según la longitud.

In [24]:
# Seleccionar un embargo corto, uno promedio y uno largo según cantidad de tokens

tokens_min = df_embargos["gliner_tokens"].min()
tokens_max = df_embargos["gliner_tokens"].max()
tokens_mediana = df_embargos["gliner_tokens"].median()

embargo_corto = df_embargos.loc[
    df_embargos["gliner_tokens"].idxmin()
]

embargo_largo = df_embargos.loc[
    df_embargos["gliner_tokens"].idxmax()
]

embargo_promedio = df_embargos.iloc[
    (df_embargos["gliner_tokens"] - tokens_mediana).abs().argsort().iloc[0]
]

df_prueba_longitud = pd.DataFrame([
    embargo_corto,
    embargo_promedio,
    embargo_largo
])

df_prueba_longitud[["id", "nombre", "clasificacion", "gliner_tokens", "largo_limpio"]]

,id,nombre,clasificacion,gliner_tokens,largo_limpio
509,542136,Embargo - usuario,Embargo,149,285
588,544859,Embargo - usuario,Embargo,1055,3448
35,496830,Embargo - usuario,Embargo,3517,10926


Después corré GLiNER sobre esos 3:

In [31]:
def extraer_entidades_para_prueba(row, labels):
    text = str(row["texto_limpio"])

    entities = model.predict_entities(text, labels)

    resultados = []

    if len(entities) == 0:
        return [{
            "id": row["id"],
            "nombre": row["nombre"],
            "clasificacion": row["clasificacion"],
            "gliner_tokens": row["gliner_tokens"],
            "largo_limpio": row["largo_limpio"],
            "entidad_texto": None,
            "entidad_label": "SIN_ENTIDADES",
            "score": None
        }]

    for ent in entities:
        resultados.append({
            "id": row["id"],
            "nombre": row["nombre"],
            "clasificacion": row["clasificacion"],
            "gliner_tokens": row["gliner_tokens"],
            "largo_limpio": row["largo_limpio"],
            "entidad_texto": ent.get("text"),
            "entidad_label": ent.get("label"),
            "score": ent.get("score")
        })

    return resultados

In [32]:
resultados_longitud = []

for _, row in df_prueba_longitud.iterrows():
    resultados_longitud.extend(
        extraer_entidades_para_prueba(row, labels_embargo)
    )

df_entidades_por_longitud = pd.DataFrame(resultados_longitud)

df_entidades_por_longitud

/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 734 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 2542 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


,id,nombre,clasificacion,gliner_tokens,largo_limpio,entidad_texto,entidad_label,score
0,542136,Embargo - usuario,Embargo,149,285,None,SIN_ENTIDADES,NaN
1,544859,Embargo - usuario,Embargo,1055,3448,CHEN FANG,persona embargada,0.596866
2,544859,Embargo - usuario,Embargo,1055,3448,Expte. 18132/2025,expediente,0.769912
3,544859,Embargo - usuario,Embargo,1055,3448,Juzgado Federal de Ejecuciones Fiscales Tribut...,juzgado,0.943470
4,544859,Embargo - usuario,Embargo,1055,3448,Dra. Patricia A.\nROTA,persona embargada,0.506305
5,544859,Embargo - usuario,Embargo,1055,3448,CHEN FANG,persona embargada,0.654965
6,544859,Embargo - usuario,Embargo,1055,3448,27-95308307-3,CUIT,0.787128
7,544859,Embargo - usuario,Embargo,1055,3448,CUIT N* 30-70308853-4,CUIT,0.541548
8,544859,Embargo - usuario,Embargo,1055,3448,MERCADOLIBRE S.R.L.,empresa embargada,0.520894
9,544859,Embargo - usuario,Embargo,1055,3448,CUIT N* 30-\n70308853-4,CUIT,0.624830


El mensaje:

```
Sentence of length 734 has been truncated to 384
Sentence of length 2542 has been truncated to 384
```

significa que GLiNER encontró fragmentos/oraciones demasiado largas y las recortó internamente a 384.

Es decir:

GLiNER no procesó completo ese fragmento largo. Lo cortó.

Esto es clave porque puede perder entidades que estén después del token 384 de ese fragmento. Por ejemplo, si el CBU, monto o persona embargada aparece más adelante, puede no detectarlo.

Resumen por documento:

In [27]:
df_entidades_por_longitud.groupby("id").agg(
    tokens=("gliner_tokens", "first"),
    entidades_detectadas=("entidad_texto", "count"),
    score_promedio=("score", "mean"),
    score_minimo=("score", "min"),
    score_maximo=("score", "max")
).sort_values("tokens")

,tokens,entidades_detectadas,score_promedio,score_minimo,score_maximo
id,,,,,
544859,1055,12,0.716379,0.506305,0.943470
496830,3517,8,0.663220,0.543861,0.904166


Y por etiqueta:

In [28]:
df_entidades_por_longitud.groupby(["id", "entidad_label"]).agg(
    cantidad=("entidad_texto", "count"),
    score_promedio=("score", "mean")
).reset_index().sort_values(["id", "cantidad"], ascending=[True, False])

,id,entidad_label,cantidad,score_promedio
4,496830,monto,3,0.620276
3,496830,juzgado,2,0.616320
0,496830,CVU,1,0.753117
1,496830,expediente,1,0.904166
2,496830,fecha,1,0.555013
5,544859,CUIT,3,0.651169
11,544859,persona embargada,3,0.586045
6,544859,banco,1,0.854469
7,544859,empresa embargada,1,0.520894
8,544859,expediente,1,0.769912


PROBAMOS PORQUE NO APARECE INFORMACION DEL EMBARGO CORTO

Lo más probable es que el embargo corto no aparezca en df_entidades_por_longitud porque GLiNER no detectó ninguna entidad en ese documento. Como tu DataFrame de resultados se arma solo con entidades detectadas, si un documento no detecta nada, no aparece.

In [29]:
df_prueba_longitud[["id", "nombre", "gliner_tokens", "largo_limpio"]]

,id,nombre,gliner_tokens,largo_limpio
509,542136,Embargo - usuario,149,285
588,544859,Embargo - usuario,1055,3448
35,496830,Embargo - usuario,3517,10926


In [30]:
ids_prueba = set(df_prueba_longitud["id"])
ids_con_entidades = set(df_entidades_por_longitud["id"])

print("IDs probados:", ids_prueba)
print("IDs con entidades detectadas:", ids_con_entidades)
print("IDs sin entidades detectadas:", ids_prueba - ids_con_entidades)

IDs probados: {542136, 544859, 496830}
IDs con entidades detectadas: {544859, 496830}
IDs sin entidades detectadas: {542136}


**El embargo corto no tuvo entidades detectadas**

Como tiene solo 285 caracteres, probablemente no contiene suficiente contexto legal para detectar cosas como monto, CBU, juzgado, expediente, etc.

In [7]:
row_corto = df_embargos[df_embargos["id"] == 542136].iloc[0]

print(row_corto["texto_limpio"])

q a ZA mercad
A Ó-—
tArñ20190D 198

POMADA

40 007. vol sh oben Wisnewslal ssobañb" Wbiscineua y lorinoo lo

Bm 3 me DATE bs D ADA

ln MOO .1 7 alansoro.o
dm y o PrRMOONS 02 |

r ' As
1
4 |
a
1
y
!
y €
S á .
ÑN
ba 5 ' |
Í
4
1]
|
otr y" i |
Sl Na?
|
, '
1. A |
|
rra NO] | |
yiv! 31 | |


El embargo corto seleccionado para la prueba tenía 149 tokens y 285 caracteres, pero al revisar su contenido se observó que el OCR era ilegible. El texto no contenía entidades legales reconocibles como expediente, juzgado, persona embargada, monto, CUIT, CBU o banco. Por ese motivo, GLiNER no detectó entidades.

Este caso muestra que la longitud del texto no es suficiente para determinar si un documento es útil. Además de textos vacíos, errores de descarga o textos muy cortos, es necesario considerar una categoría de baja calidad OCR para textos que tienen contenido, pero no son interpretables.

## Conclusión de la prueba con GLiNER por longitud de documento

Se realizó una prueba inicial de extracción de entidades con GLiNER sobre embargos de distinta longitud, tomando como referencia la cantidad de tokens calculada con el tokenizer del modelo.

En la exploración general del dataset de embargos se observó que existen 627 documentos, con una media de 1128 tokens, un mínimo de 149 tokens y un máximo de 3517 tokens. Esto indica que la mayoría de los embargos tiene una longitud manejable, aunque existen casos considerablemente más extensos.

En la prueba por longitud, GLiNER detectó entidades relevantes en documentos de tamaño medio y largo, como expediente, juzgado, CUIT, montos, banco, fecha, plazo y persona embargada. Sin embargo, durante la ejecución aparecieron advertencias indicando que algunas secuencias fueron truncadas a 384. Esto significa que GLiNER no procesó completamente ciertos fragmentos largos del texto.

El documento de tamaño medio analizado presentó 1055 tokens, 12 entidades detectadas y un score promedio aproximado de 0.716. El documento largo presentó 3517 tokens, 8 entidades detectadas y un score promedio aproximado de 0.663. Esta diferencia sugiere que, en textos más largos, la calidad de extracción puede disminuir y existe riesgo de pérdida de entidades por truncamiento.

Como conclusión, GLiNER resulta útil para una primera extracción de entidades en embargos, pero antes de aplicarlo sobre la totalidad de los documentos conviene implementar o probar una estrategia de división del texto en fragmentos más pequeños. Esto permitiría evitar truncamientos y mejorar la cobertura de entidades en documentos largos.


## Conclusión de la prueba corto / promedio / largo

Se probaron tres embargos con distinta longitud para observar cómo responde GLiNER según la cantidad de tokens.

El embargo corto, con 149 tokens y 285 caracteres, no generó entidades detectadas. Esto puede deberse a que el texto es demasiado breve, contiene poca información legal útil o no incluye entidades compatibles con las etiquetas definidas.

El embargo de longitud media, con 1055 tokens, generó 12 entidades detectadas y un score promedio de aproximadamente 0.716. Este resultado indica que GLiNER puede funcionar razonablemente bien en documentos de tamaño medio.

El embargo largo, con 3517 tokens, generó 8 entidades detectadas y un score promedio de aproximadamente 0.663. Sin embargo, durante la ejecución apareció una advertencia de truncamiento, indicando que una secuencia larga fue recortada internamente a 384. Esto implica riesgo de pérdida de información en documentos extensos.

Como conclusión, GLiNER muestra resultados útiles para una primera extracción de entidades en embargos, pero los documentos largos requieren una estrategia adicional, como dividir el texto en fragmentos más pequeños antes de procesarlo.